In [1]:
# ============================================================
# Geo-FNO training on cylinder_flow H5 (predict velocity; input includes pressure, no node-type channel)
# - Extract cylinder (xc, yc, r) from node_type==6 by removing
#   top/bottom boundary bands in y, then circle-fit (Kåsa LS).
# - Feed (xc, yc, r) into the 42-dim "code" vector.
# - Modify IPHI so its center comes from code[:,0:2] (critical).
# - Train one-step: vel[t_in] -> vel[t_out]
# ============================================================

import os
import math
import h5py
import numpy as np
from typing import Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim import Adam
from geo_FNO_def import *
import torch.nn as nn

# ----------------------------
# 1) Cylinder fit utilities
# ----------------------------

def fit_circle_kasa(x: np.ndarray, y: np.ndarray) -> Tuple[float, float, float]:
    """Least-squares circle fit. Returns (xc, yc, r)."""
    A = np.stack([x, y, np.ones_like(x)], axis=1)      # (M,3)
    b = -(x**2 + y**2)                                 # (M,)
    sol, *_ = np.linalg.lstsq(A, b, rcond=None)
    a, b_, c = sol
    xc = -a / 2.0
    yc = -b_ / 2.0
    r2 = (a*a + b_*b_) / 4.0 - c
    r = float(np.sqrt(max(r2, 0.0)))
    return float(xc), float(yc), float(r)

def estimate_cylinder_from_label6(
    pos: np.ndarray,
    node_type: np.ndarray,
    boundary_label: int = 6,
    band_frac: float = 0.06,
) -> Tuple[float, float, float]:
    """
    node_type==6 contains walls + cylinder. Remove top/bottom walls by y bands,
    fit circle to remaining points.
    """
    xy = pos[node_type == boundary_label]
    if xy.shape[0] < 20:
        raise RuntimeError(f"Too few nodes with node_type=={boundary_label}")

    y = xy[:, 1]
    ymin, ymax = float(y.min()), float(y.max())
    H = ymax - ymin
    band = band_frac * H

    cyl_mask = (y > ymin + band) & (y < ymax - band)
    xy_cyl = xy[cyl_mask]
    if xy_cyl.shape[0] < 20:
        raise RuntimeError(
            f"After wall-band filtering, too few points to fit circle. "
            f"Try smaller band_frac. points={xy_cyl.shape[0]}"
        )

    xc, yc, r = fit_circle_kasa(xy_cyl[:, 0], xy_cyl[:, 1])
    return xc, yc, r


# ----------------------------
# 2) Dataset (H5) -> (x_in, u_in, u_out, code42)
# ----------------------------

class CylinderFlowGeoFNODataset(Dataset):
    """
    Returns one sample as:
      x_in  : (N,2) node coordinates
      u_in  : (N,3) [velocity at t_in, pressure at t_in]
      u_out : (N,2) velocity at t_out
      code  : (42,) code vector with [xc, yc, r] in first 3 entries
    """
    def __init__(
        self,
        h5_path: str,
        t_in: int = 0,
        t_out: int = 200,
        boundary_label: int = 6,
        band_frac: float = 0.06,
        cache_geom: bool = True,
        node_type_values = None,
    ):
        super().__init__()
        self.f = h5py.File(h5_path, "r")
        self.keys = sorted([k for k in self.f.keys() if k.startswith("sample_")])
        self.t_in = t_in
        self.t_out = t_out
        self.boundary_label = boundary_label
        self.band_frac = band_frac
        self.cache_geom = cache_geom
        self._geom_cache = {}  # key -> (xc,yc,r)

        if node_type_values is None:
            uniq = set()
            for k in self.keys:
                nt = self.f[k]["node_type"][:]
                uniq.update(int(v) for v in np.unique(nt))
            self.node_type_values = sorted(list(uniq))
        else:
            self.node_type_values = [int(v) for v in node_type_values]

        self.node_type_to_idx = {v: i for i, v in enumerate(self.node_type_values)}
        self.num_node_type_channels = len(self.node_type_values)

    def __len__(self):
        return len(self.keys)

    def _get_geom(self, key: str) -> Tuple[float, float, float]:
        if self.cache_geom and key in self._geom_cache:
            return self._geom_cache[key]
        g = self.f[key]
        pos = g["pos"][:]                # (N,2)
        L1 = pos[:,0].max() - pos[:,0].min()
        L2 = pos[:,1].max() - pos[:,1].min()
        node_type = g["node_type"][:]    # (N,)
        xc, yc, r = estimate_cylinder_from_label6(
            pos, node_type, boundary_label=self.boundary_label, band_frac=self.band_frac
        )
        if self.cache_geom:
            self._geom_cache[key] = (xc, yc, r)
        return xc, yc, r

    def __getitem__(self, idx: int):
        key = self.keys[idx]
        g = self.f[key]
        pos = g["pos"][:]                 # (N,2)
        vel = g["vel"][:]                 # (T,N,2)
        pressure = g["pressure"][:]       # (T,N)
        node_type = g["node_type"][:].astype(np.int32)  # (N,)

        u_vel = vel[self.t_in].astype(np.float32)                 # (N,2)
        p_in = pressure[self.t_in].astype(np.float32)[:, None]    # (N,1)
        u_out = vel[self.t_out].astype(np.float32)                # (N,2)

        u_in = np.concatenate([u_vel, p_in], axis=-1)  # (N, 3)

        xc, yc, r = self._get_geom(key)

        code = np.zeros((42,), dtype=np.float32)
        code[0] = xc
        code[1] = yc
        code[2] = r

        return (
            torch.from_numpy(pos.astype(np.float32)),   # (N,2)
            torch.from_numpy(u_in),                     # (N,2+K)
            torch.from_numpy(u_out),                    # (N,2)
            torch.from_numpy(code),                     # (42,)
        )

def collate_bs1(batch):
    # simplest: use batch_size=1 initially
    return batch[0]

def compute_io_stats_from_indices(ds, indices):
    """Compute weighted per-component stats over all nodes in selected samples."""
    in_ch = 3
    sum_in = torch.zeros(in_ch, dtype=torch.float64)
    sq_in = torch.zeros(in_ch, dtype=torch.float64)
    n_in = 0

    sum_out = torch.zeros(2, dtype=torch.float64)
    sq_out = torch.zeros(2, dtype=torch.float64)
    n_out = 0

    for i in indices:
        _, u_in, u_out, _ = ds[i]
        u_in64 = u_in.to(torch.float64)
        u_out64 = u_out.to(torch.float64)

        sum_in += u_in64.sum(dim=0)
        sq_in += (u_in64 * u_in64).sum(dim=0)
        n_in += int(u_in64.shape[0])

        sum_out += u_out64.sum(dim=0)
        sq_out += (u_out64 * u_out64).sum(dim=0)
        n_out += int(u_out64.shape[0])

    mean_in = sum_in / max(1, n_in)
    mean_out = sum_out / max(1, n_out)

    var_in = torch.clamp(sq_in / max(1, n_in) - mean_in * mean_in, min=1e-12)
    var_out = torch.clamp(sq_out / max(1, n_out) - mean_out * mean_out, min=1e-12)

    std_in = torch.sqrt(var_in)
    std_out = torch.sqrt(var_out)

    return {
        "u_in_mean": mean_in.to(torch.float32),
        "u_in_std": std_in.to(torch.float32),
        "u_out_mean": mean_out.to(torch.float32),
        "u_out_std": std_out.to(torch.float32),
    }



def train_geofno_cylinder(
    train_h5: str,
    test_h5: str,
    ntrain: int = 1000,
    ntest: int = 200,
    t_in: int = 0,
    t_out: int = 200,
    boundary_label: int = 6,
    band_frac: float = 0.06,
    batch_size: int = 1,
    epochs: int = 201,
    learning_rate_fno: float = 1e-3,
    learning_rate_iphi: float = 1e-4,
    modes1: int = 12,
    modes2: int = 12,
    width: int = 32,
    s1: int = 40,
    s2: int = 40,
    device: str = "cuda:0",
    # ---- NEW ----
    val_patience: int = 25,
    improve_thresh: float = 1e-3,
    use_io_normalization: bool = False,
):
    # Data
    train_ds = CylinderFlowGeoFNODataset(
        train_h5, t_in=t_in, t_out=t_out, boundary_label=boundary_label, band_frac=band_frac, cache_geom=True
    )
    test_ds = CylinderFlowGeoFNODataset(
        test_h5, t_in=t_in, t_out=t_out, boundary_label=boundary_label, band_frac=band_frac,
        cache_geom=True, node_type_values=train_ds.node_type_values
    )

    train_loader = DataLoader(
        Subset(train_ds, range(min(ntrain, len(train_ds)))),
        batch_size=batch_size, shuffle=True, collate_fn=collate_bs1
    )

    subset_indices = train_loader.dataset.indices  # Subset stores indices here
    print("Training subset indices:", subset_indices)
    print("Training sample keys:", [train_ds.keys[i] for i in subset_indices])

    test_loader = DataLoader(
        Subset(test_ds, range(min(ntest, len(test_ds)))),
        batch_size=batch_size, shuffle=False, collate_fn=collate_bs1
    )

    L_global, key_used = get_global_L_from_h5(train_h5)
    print("Using global L from", key_used, ":", L_global)

    # Models
    in_channels = 3
    print("Input channels (vel + pressure):", in_channels)
    model = FNO2d(modes1, modes2, width, in_channels=in_channels, out_channels=2, is_mesh=False, s1=s1, s2=s2, L = L_global).to(device)
    model_iphi = IPHI(width=32, device=device).to(device)

    opt_fno = Adam(model.parameters(), lr=learning_rate_fno, weight_decay=1e-4)
    opt_iphi = Adam(model_iphi.parameters(), lr=learning_rate_iphi, weight_decay=1e-4)
    sched_fno = torch.optim.lr_scheduler.CosineAnnealingLR(opt_fno, T_max=epochs)
    sched_iphi = torch.optim.lr_scheduler.CosineAnnealingLR(opt_iphi, T_max=epochs)

    if use_io_normalization:
        stats = compute_io_stats_from_indices(train_ds, subset_indices)
        model.set_io_normalization(
            stats["u_in_mean"], stats["u_in_std"],
            stats["u_out_mean"], stats["u_out_std"],
            enabled=True,
        )
        print("I/O normalization enabled")
        print("  u_in  mean/std:", stats["u_in_mean"].tolist(), stats["u_in_std"].tolist())
        print("  u_out mean/std:", stats["u_out_mean"].tolist(), stats["u_out_std"].tolist())
    else:
        model.set_io_normalization_enabled(False)
        print("I/O normalization disabled")

    loss_fn = nn.MSELoss()

    #Early stopping implemented
    best_val_loss = 1e12
    epochs_no_improve = 0
    best_state_model = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    best_state_iphi  = {k: v.detach().cpu().clone() for k, v in model_iphi.state_dict().items()}

    for ep in range(epochs):
        # ---- train ----
        model.train()
        model_iphi.train()
        train_loss = 0.0

        for pos, u_in, u_out, code42 in train_loader:
            # add batch dim
            pos = pos.unsqueeze(0).to(device)        # (1,N,2)
            u_in = u_in.unsqueeze(0).to(device)      # (1,N,2)
            u_out = u_out.unsqueeze(0).to(device)    # (1,N,2)
            code42 = code42.unsqueeze(0).to(device)  # (1,42)

            opt_fno.zero_grad()
            opt_iphi.zero_grad()

            pred = model(u_in, code=code42, x_in=pos, x_out=pos, iphi=model_iphi)  # (1,N,2)
            loss = loss_fn(pred, u_out)
            loss.backward()

            opt_fno.step()
            opt_iphi.step()

            train_loss += float(loss.item())

        train_loss /= max(1, len(train_loader))

        # ---- eval (this is your "val") ----
        model.eval()
        model_iphi.eval()
        val_loss = 0.0
        with torch.no_grad():
            for pos, u_in, u_out, code42 in test_loader:
                pos = pos.unsqueeze(0).to(device)
                u_in = u_in.unsqueeze(0).to(device)
                u_out = u_out.unsqueeze(0).to(device)
                code42 = code42.unsqueeze(0).to(device)

                pred = model(u_in, code=code42, x_in=pos, x_out=pos, iphi=model_iphi)
                val_loss += float(loss_fn(pred, u_out).item())

        val_loss /= max(1, len(test_loader))
        sched_fno.step()
        sched_iphi.step()

        print(f"ep={ep:04d} train={train_loss:.6e} val={val_loss:.6e}")

        #Early stopping
        if val_loss < best_val_loss * (1 - improve_thresh):
            best_val_loss = val_loss
            best_state_model = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_state_iphi  = {k: v.detach().cpu().clone() for k, v in model_iphi.state_dict().items()}
            epochs_no_improve = 0
            
        else:
            if best_val_loss * (1 - improve_thresh) < val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state_model = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                best_state_iphi  = {k: v.detach().cpu().clone() for k, v in model_iphi.state_dict().items()}

            epochs_no_improve += 1
            if epochs_no_improve >= val_patience:
                print("Early stop triggered.")
                break

        # save checkpoints occasionally (unchanged)
        if ep % 100 == 0:
            os.makedirs("./checkpoints", exist_ok=True)
            torch.save(model.state_dict(), f"./checkpoints/geofno_vel_ep{ep}.pt")
            torch.save(model_iphi.state_dict(), f"./checkpoints/iphi_ep{ep}.pt")

    # ---- NEW: load best weights before returning ----
    model.load_state_dict(best_state_model)
    model_iphi.load_state_dict(best_state_iphi)
    print("Best val loss:", best_val_loss)

    return model, model_iphi


In [3]:
train_h5 = "/scratch/mnhagen/datasets/incompressible_euler/train.h5"
test_h5  = "/scratch/mnhagen/datasets/incompressible_euler/test.h5"

device = "cuda:0" if torch.cuda.is_available() else "cpu"

# This variant feeds vel + pressure[t_in] (no node_type channel) into Geo-FNO
model, model_iphi = train_geofno_cylinder(
    train_h5=train_h5,
    test_h5=test_h5,
    ntrain=1000,
    ntest=200,
    t_in=0,
    t_out=-1,
    boundary_label=6,
    band_frac=0.06,          # tweak if needed
    batch_size=1,            # keep 1 to avoid variable-N batching issues initially
    epochs=1000,
    learning_rate_fno=1e-3,
    learning_rate_iphi=1e-4,
    modes1=24,
    modes2 = 12,
    width=32,
    s1=100,
    s2=25,
    device=device,
    val_patience = 25,
    use_io_normalization=True
)

Training subset indices: range(0, 1000)
Training sample keys: ['sample_000000', 'sample_000001', 'sample_000002', 'sample_000003', 'sample_000004', 'sample_000005', 'sample_000006', 'sample_000007', 'sample_000008', 'sample_000009', 'sample_000010', 'sample_000011', 'sample_000012', 'sample_000013', 'sample_000014', 'sample_000015', 'sample_000016', 'sample_000017', 'sample_000018', 'sample_000019', 'sample_000020', 'sample_000021', 'sample_000022', 'sample_000023', 'sample_000024', 'sample_000025', 'sample_000026', 'sample_000027', 'sample_000028', 'sample_000029', 'sample_000030', 'sample_000031', 'sample_000032', 'sample_000033', 'sample_000034', 'sample_000035', 'sample_000036', 'sample_000037', 'sample_000038', 'sample_000039', 'sample_000040', 'sample_000041', 'sample_000042', 'sample_000043', 'sample_000044', 'sample_000045', 'sample_000046', 'sample_000047', 'sample_000048', 'sample_000049', 'sample_000050', 'sample_000051', 'sample_000052', 'sample_000053', 'sample_000054', 's

In [4]:
import os, json
import torch

def save_geofno_checkpoint(
    model,
    model_iphi,
    folder: str,
    name: str,
    extra_meta: dict | None = None,
):
    os.makedirs(folder, exist_ok=True)
    model_path = os.path.join(folder, f"{name}_fno.pt")
    iphi_path  = os.path.join(folder, f"{name}_iphi.pt")
    meta_path  = os.path.join(folder, f"{name}_meta.json")

    torch.save(model.state_dict(), model_path)
    torch.save(model_iphi.state_dict(), iphi_path)

    meta = {} if extra_meta is None else dict(extra_meta)
    meta["model_path"] = model_path
    meta["iphi_path"] = iphi_path
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    print("Saved:", model_path)
    print("Saved:", iphi_path)
    print("Saved:", meta_path)

save_geofno_checkpoint(model, model_iphi,
                       folder="/scratch/mnhagen/models/geofno",
                       name="cylinder_vel_t0_t-1_with_pressure_asymmetrical",
                       extra_meta={"t_in": 0, "t_out": -1, "boundary_label": 6, "band_frac": 0.06})

Saved: /scratch/mnhagen/models/geofno/cylinder_vel_t0_t-1_with_pressure_asymmetrical_fno.pt
Saved: /scratch/mnhagen/models/geofno/cylinder_vel_t0_t-1_with_pressure_asymmetrical_iphi.pt
Saved: /scratch/mnhagen/models/geofno/cylinder_vel_t0_t-1_with_pressure_asymmetrical_meta.json
